In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from datetime import datetime
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import statsmodels.api as sm
import os

In [ ]:
ahead = pd.read_csv('../data/ahead.csv')
times = pd.read_csv('../data/times.csv')

del ahead['num_cores']

df = pd.merge(times, ahead, on='file')

In [ ]:
print(df.shape)
print(ahead.shape)
print(times.shape)

In [ ]:
def unix_to_date(unix_timestamp):
    dt = datetime.fromtimestamp(unix_timestamp)
    formatted_date = dt.strftime("%a %b %d %I:%M:%S %p %Y")
    return formatted_date

In [ ]:
def am_pm_label(num):
    day_or_night = "AM"
    if num >= 12:
        day_or_night = "PM"
    
    digit = num%12
    if digit == 0:
        digit += 12
  
    return (f"{digit} {day_or_night}")

In [ ]:
if os.path.isdir('figs') == False:
    os.mkdir('figs')

In [ ]:
df['queuing_delay'] = df['started'] - df['submitted']
df['submitted_dt'] = pd.to_datetime(df['submitted'], unit='s') - pd.Timedelta('04:00:00')
df['started_dt'] = pd.to_datetime(df['started'], unit='s') - pd.Timedelta('04:00:00')

# Sort by submission time
df = df.sort_values('submitted_dt')

In [ ]:
df_no_outliers = df.copy()

for num_cores in [4, 16, 64]:
    mask = df_no_outliers['num_cores'] == num_cores
    subset = df_no_outliers.loc[mask, 'queuing_delay']
    
    Q1 = subset.quantile(0.25)
    Q3 = subset.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Keep only rows within bounds for this num_cores group
    valid = (subset >= lower_bound) & (subset <= upper_bound)
    df_no_outliers = df_no_outliers[~mask | valid]

# df = df_no_outliers

In [ ]:
print(f"Total measurements: {len(df[df['num_cores'] == 4])}")
print(f"Time period: {df[df['num_cores'] == 4]['submitted_dt'].min()} to {df[df['num_cores'] == 4]['submitted_dt'].max()}")
print(f"Duration: {(df[df['num_cores'] == 4]['submitted_dt'].max() - df[df['num_cores'] == 4]['submitted_dt'].min()).days} days")

for num_cores in [4, 16, 64]:
    print(f"\n=== Queuing Delay Summary Statistics for {num_cores} cores ===")
    print(f"Queuing Delay (seconds):")
    print(f"  Mean: {df[df['num_cores'] == num_cores]['queuing_delay'].mean():.2f}")
    print(f"  Median: {df[df['num_cores'] == num_cores]['queuing_delay'].median():.2f}")
    print(f"  Min: {df[df['num_cores'] == num_cores]['queuing_delay'].min():.2f}")
    print(f"  Max: {df[df['num_cores'] == num_cores]['queuing_delay'].max():.2f}")
    print(f"  Std Dev: {df[df['num_cores'] == num_cores]['queuing_delay'].std():.2f}")
    print(f"\nJobs in Queue:")
    print(f"  Mean: {df[df['num_cores'] == num_cores]['jobs_ahead'].mean():.2f}")
    print(f"  Max: {df[df['num_cores'] == num_cores]['jobs_ahead'].max()}")

    num_cores *= 4

In [ ]:
for num_cores in [4, 16, 64]:
    print(f"\n=== Queuing Delay Summary Statistics for {num_cores} cores ===")
    new_df = df[df['num_cores'] == num_cores]

    max_delay_row = new_df.loc[new_df['queuing_delay'].idxmax()]
    min_delay_row = new_df.loc[new_df['queuing_delay'].idxmin()]

    print(f"Submitted longest waiting at: {unix_to_date(max_delay_row.submitted)}")
    print(f"Longest waiting started running at: {unix_to_date(max_delay_row.started)}")
    print(f"Waited for {max_delay_row.started - max_delay_row.submitted:.2f} seconds \n")

    print(f"Submitted shortest waiting at: {unix_to_date(min_delay_row.submitted)}")
    print(f"Shortest waiting started running at: {unix_to_date(min_delay_row.started)}")
    print(f"Waited for {min_delay_row.started - min_delay_row.submitted:.2f} seconds")

In [ ]:
for num_cores in [4, 16, 64]:
    new_df = df[df['num_cores'] == num_cores]

    # Create the plot
    fig, ax1 = plt.subplots(figsize=(14, 7))

    # Plot queuing delay over time
    color = 'tab:blue'
    ax1.set_xlabel('Date and Time', fontsize=12)
    ax1.set_ylabel('Queuing Delay (seconds)', fontsize=12, color=color)
    ax1.plot(new_df['submitted_dt'], new_df['queuing_delay'], 
            color=color, linewidth=3, marker='o', markersize=2, alpha=0.7)
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(True, alpha=0.3)

    # Create a second y-axis for number of jobs in queue
    ax2 = ax1.twinx()
    color = 'tab:orange'
    ax2.set_ylabel('Number of Jobs in Queue', fontsize=12, color=color)
    ax2.plot(new_df['submitted_dt'], new_df['jobs_ahead'], 
            color=color, linewidth=1.5, marker='s', markersize=2, alpha=0.5)
    ax2.tick_params(axis='y', labelcolor=color)

    # Format x-axis to show dates nicely with 30-minute intervals
    ax1.xaxis.set_major_locator(mdates.MinuteLocator(interval=240))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

    # Add title and legend
    plt.title(f'UGE Cluster Queuing Delay Over Time for {num_cores} Core Jobs', fontsize=14, fontweight='bold')
    ax1.legend(['Queuing Delay'], loc='upper left')
    ax2.legend(['Jobs in Queue'], loc='upper right')

    plt.tight_layout()
    plt.savefig(f'figs/Queuing Delay vs. Time {num_cores} Cores.pdf', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Create the plot
fig, ax1 = plt.subplots(figsize=(14, 7))

# Plot queuing delay over time
color = 'tab:blue'
ax1.set_xlabel('Date and Time', fontsize=12)
ax1.set_ylabel('Jobs in Queue', fontsize=12, color=color)
ax1.plot(new_df['submitted_dt'], new_df['jobs_in_my_queue'], 
        color=color, linewidth=1.5, marker='o', markersize=2, alpha=0.7)
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)

# Create a second y-axis for number of jobs in queue
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Jobs Running', fontsize=12, color=color)
ax2.plot(new_df['submitted_dt'], new_df['jobs_running'], 
        color=color, linewidth=1.5, marker='s', markersize=2, alpha=0.5)
ax2.tick_params(axis='y', labelcolor=color)

# Format x-axis to show dates nicely with 30-minute intervals
ax1.xaxis.set_major_locator(mdates.MinuteLocator(interval=400))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add title and legend
plt.title(f'Queuing Analysis over Time', fontsize=14, fontweight='bold')
ax1.legend(['Queuing Delay'], loc='upper left')
ax2.legend(['Jobs in Queue'], loc='upper right')

plt.tight_layout()
plt.savefig(f'figs/Jobs in Queue and Jobs Running over Time.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df['hour'] = df['submitted_dt'].dt.hour
df['day_of_week'] = df['submitted_dt'].dt.day_name()
df['day_num'] = df['submitted_dt'].dt.dayofweek  # Monday=0, Sunday=6

# Create pivot table for heatmap
# Average queuing delay by day of week and hour
heatmap_data = df.pivot_table(
    values='queuing_delay',
    index='day_of_week',
    columns='hour',
    aggfunc='mean'
)

# Reorder days to start with Monday
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = heatmap_data.reindex([day for day in day_order if day in heatmap_data.index])

# Create the heatmap
plt.figure(figsize=(16, 8))
sns.heatmap(
    heatmap_data,
    cmap='YlOrRd',
    annot=True,
    fmt='.1f',
    cbar_kws={'label': 'Average Queuing Delay (seconds)'},
    linewidths=0.5,
    linecolor='gray',
    annot_kws={'rotation': 90, 'fontsize': 9}
)

plt.title('Average Queuing Delay by Day of Week and Hour of Day', fontsize=16, pad=20)
plt.xlabel('Hour of Day', fontsize=12)
plt.ylabel('Day of Week', fontsize=12)
plt.xticks(ticks = np.arange(0,24)+.5, labels = [am_pm_label(i) for i in range(24)])
plt.tight_layout()
plt.savefig(f'figs/Heatmap of Average Queuing Delay by Day of Week and Hour of Day.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
# print("\nQueuing Delay Statistics by Hour:")
# print(df.groupby('hour')['queuing_delay'].agg(['mean', 'std', 'count']))

# print("\nQueuing Delay Statistics by Day of Week:")
# print(df.groupby('day_of_week')['queuing_delay'].agg(['mean', 'std', 'count']))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 6))

for idx, cores in enumerate([4, 16, 64]):
    df_cores = df[df['num_cores'] == cores]
    heatmap_cores = df_cores.pivot_table(
        values='queuing_delay',
        index='day_of_week',
        columns='hour',
        aggfunc='mean'
    )
    heatmap_cores = heatmap_cores.reindex([day for day in day_order if day in heatmap_cores.index])
    
    sns.heatmap(
        heatmap_cores,
        cmap='YlOrRd',
        annot=True,
        fmt='.1f',
        cbar_kws={'label': 'Avg Delay (s)'},
        linewidths=0.5,
        linecolor='gray',
        ax=ax[idx],
        annot_kws={'rotation': 90, 'fontsize': 9}
    )
    ax[idx].set_title(f'{cores} Cores', fontsize=14)
    ax[idx].set_xlabel('Hour of Day')
    ax[idx].set_ylabel('Day of Week' if idx == 0 else '')

plt.suptitle('Average Queuing Delay by Day, Hour, and Core Count', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(f'figs/Heatmap of Average Queuing Delay by Day, Hour, and Core Count.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
for cores in [4, 16, 64]:
    df_cores = df[df['num_cores'] == cores]
    heatmap_cores = df_cores.pivot_table(
        values='queuing_delay',
        index='day_of_week',
        columns='hour',
        aggfunc='mean'
    )
    heatmap_cores = heatmap_cores.reindex([day for day in day_order if day in heatmap_cores.index])

    # Create a new figure for each core count
    plt.figure(figsize=(10, 6))  # Adjust size as needed

    sns.heatmap(
        heatmap_cores,
        cmap='YlOrRd',
        annot=True,
        fmt='.1f',
        cbar_kws={'label': 'Avg Delay (s)'},
        linewidths=0.5,
        linecolor='gray',
        annot_kws={'rotation': 90, 'fontsize': 9}
    )

    plt.title(f'Average Queuing Delay - {cores} Cores', fontsize=14)
    plt.xlabel('Hour of Day')
    plt.xticks(ticks = np.arange(0,24)+.5, labels = [am_pm_label(i) for i in range(24)], rotation=45)

    plt.ylabel('Day of Week')

    plt.tight_layout()
    plt.savefig(f'figs/Heatmap_Avg_Queuing_Delay_{cores}_Cores.pdf', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Create heatmap showing average delay by hour (combining all days)
fig, ax = plt.subplots(1, 3, figsize=(20, 6))

for idx, cores in enumerate([4, 16, 64]):
    df_cores = df[df['num_cores'] == cores]
    
    # Group by hour only (combining all days)
    hourly_avg = df_cores.groupby('hour')['queuing_delay'].agg(['mean', 'count']).reset_index()
    
    # Create a 1-row heatmap (reshape for heatmap format)
    heatmap_data = hourly_avg.pivot_table(
        values='mean',
        columns='hour',
        aggfunc='first'
    )
    
    sns.heatmap(
        heatmap_data,
        cmap='YlOrRd',
        annot=True,
        fmt='.1f',
        cbar_kws={'label': 'Avg Delay (s)'},
        linewidths=0.5,
        linecolor='gray',
        ax=ax[idx],
        yticklabels=['All Days'],
        annot_kws={'rotation': 90, 'fontsize': 9}
    )
    ax[idx].set_title(f'{cores} Cores', fontsize=14)
    ax[idx].set_xlabel('Hour of Day')
    ax[idx].set_ylabel('')

plt.suptitle('Average Queuing Delay by Hour of Day (All Days Combined)', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(f'figs/Heatmap of Average Queuing Delay by Hour of Day (All Days Combined).pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
for cores in [4, 16, 64]:
    df_cores = df[df['num_cores'] == cores]

    # Group by hour only (combining all days)
    hourly_avg = df_cores.groupby('hour')['queuing_delay'].agg(['mean', 'count']).reset_index()

    # Reshape to 1-row heatmap format
    heatmap_data = hourly_avg.pivot_table(
        values='mean',
        columns='hour',
        aggfunc='first'
    )

    # Create new figure for each core count
    plt.figure(figsize=(10, 2.5))  # Wider and shorter for 1-row heatmap

    sns.heatmap(
        heatmap_data,
        cmap='YlOrRd',
        annot=True,
        fmt='.1f',
        cbar_kws={'label': 'Avg Delay (s)'},
        linewidths=0.5,
        linecolor='gray',
        yticklabels=['All Days'],
        annot_kws={'rotation': 90, 'fontsize': 9}
    )

    plt.title(f'Average Queuing Delay by Hour - {cores} Cores', fontsize=14)
    plt.xlabel('Hour of Day')
    plt.xticks(ticks = np.arange(0,24)+.5, labels = [am_pm_label(i) for i in range(24)], rotation=45)
    
    plt.ylabel('')

    plt.tight_layout()
    plt.savefig(f'figs/Heatmap_Avg_Queuing_Delay_Hourly_{cores}_Cores.pdf', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 6))

for idx, cores in enumerate([4, 16, 64]):
    df_cores = df[df['num_cores'] == cores]
    
    # Group by hour
    hourly_stats = df_cores.groupby('hour')['queuing_delay'].agg(['mean', 'std', 'count']).reset_index()
    
    # Create bar chart
    bars = ax[idx].bar(hourly_stats['hour'], hourly_stats['mean'], 
                          color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5)
    
    # Color bars by delay (gradient from green to red)
    max_delay = hourly_stats['mean'].max()
    for bar, delay in zip(bars, hourly_stats['mean']):
        intensity = delay / max_delay if max_delay > 0 else 0
        r = intensity
        g = 1 - intensity
        bar.set_facecolor((r, g, 0.2, 0.7))
    
    # Add error bars if we have std
    if hourly_stats['std'].notna().any():
        ax[idx].errorbar(hourly_stats['hour'], hourly_stats['mean'], 
                           yerr=hourly_stats['std'], fmt='none', 
                           ecolor='black', alpha=0.3, capsize=3)
    
    ax[idx].set_title(f'{cores} Cores', fontsize=14)
    ax[idx].set_xlabel('Hour of Day', fontsize=12)
    ax[idx].set_ylabel('Average Delay (seconds)' if idx == 0 else '', fontsize=12)
    ax[idx].set_xticks(range(0, 24))
    ax[idx].grid(True, alpha=0.3, axis='y')
    
    # Highlight best time (lowest delay)
    if len(hourly_stats) > 0:
        best_hour = hourly_stats.loc[hourly_stats['mean'].idxmin(), 'hour']
        best_delay = hourly_stats['mean'].min()
        ax[idx].axvline(best_hour, color='green', linestyle='--', linewidth=2, alpha=0.5)
        ax[idx].text(best_hour, best_delay, f'  Best: {int(best_hour)}:00\n  ({best_delay:.1f}s)', 
                       fontsize=9, va='bottom', color='green', fontweight='bold')

plt.suptitle('Average Queuing Delay by Hour of Day - Bar Chart (All Days Combined)', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(f'figs/Average Queuing Delay by Hour of Day - Bar Chart (All Days Combined).pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
for num_cores in [4, 16, 64]:
    df_cores = df[df['num_cores'] == num_cores]
    
    # Group by hour
    hourly_stats = df_cores.groupby('hour')['queuing_delay'].agg(['mean', 'std', 'count']).reset_index()
    
    # Create bar chart
    bars = plt.bar(hourly_stats['hour'], hourly_stats['mean'], 
                          color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5)
    
    # Color bars by delay (gradient from green to red)
    max_delay = hourly_stats['mean'].max()
    for bar, delay in zip(bars, hourly_stats['mean']):
        intensity = delay / max_delay if max_delay > 0 else 0
        r = intensity
        g = 1 - intensity
        bar.set_facecolor((r, g, 0.2, 0.7))
    
    # Add error bars if we have std
    if hourly_stats['std'].notna().any():
        plt.errorbar(hourly_stats['hour'], hourly_stats['mean'], 
                           yerr=hourly_stats['std'], fmt='none', 
                           ecolor='black', alpha=0.3, capsize=3)
    
    plt.title(f'{num_cores} Cores', fontsize=14)
    plt.xlabel('Hour of Day', fontsize=12)
    plt.xticks(ticks = np.arange(0,24), labels = [am_pm_label(i) for i in range(24)], rotation=45)
    plt.ylabel('Average Delay (seconds)', fontsize=12)

    plt.grid(True, alpha=0.3, axis='y')
    
    # Highlight best time (lowest delay)
    if len(hourly_stats) > 0:
        best_hour = hourly_stats.loc[hourly_stats['mean'].idxmin(), 'hour']
        best_delay = hourly_stats['mean'].min()
        plt.axvline(best_hour, color='green', linestyle='--', linewidth=2, alpha=0.5)
        plt.text(best_hour, best_delay, f'  Best: {int(best_hour)}:00\n  ({best_delay:.1f}s)', 
                       fontsize=9, va='bottom', color='green', fontweight='bold')

    plt.title(f'Average Queuing Delay by Hour of Day - {num_cores} Cores', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.savefig(f'figs/Average Queuing Delay by Hour of Day - {num_cores} Cores.pdf', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
print("=" * 80)
print("QUEUE STATE IMPACT ANALYSIS")
print("=" * 80)

# Prepare features for regression
X = df[['jobs_ahead', 'jobs_running', 'num_cores']]
y = df['queuing_delay']

# Add constant for intercept
X_with_const = sm.add_constant(X)

# Fit OLS model for detailed statistics
model_ols = sm.OLS(y, X_with_const).fit()

print("\n" + "=" * 80)
print("LINEAR REGRESSION MODEL: queuing_delay ~ jobs_ahead + jobs_running + num_cores")
print("=" * 80)
print(model_ols.summary())

# Extract key metrics
print("\n" + "=" * 80)
print("KEY FINDINGS")
print("=" * 80)
print(f"\nR-squared: {model_ols.rsquared:.4f}")
print(f"Adjusted R-squared: {model_ols.rsquared_adj:.4f}")
print(f"Root Mean Squared Error: {np.sqrt(mean_squared_error(y, model_ols.predict(X_with_const))):.2f} seconds")

print("\n" + "-" * 80)
print("COEFFICIENT INTERPRETATION:")
print("-" * 80)
for var, coef, pval in zip(['Intercept', 'jobs_ahead', 'jobs_running', 'num_cores'], 
                            model_ols.params, model_ols.pvalues):
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"{var:15s}: {coef:8.2f} seconds  (p-value: {pval:.4f}) {sig}")

print("\nInterpretation:")
print("- Each additional job ahead adds ~{:.2f} seconds to delay".format(model_ols.params['jobs_ahead']))
print("- Each additional running job adds ~{:.2f} seconds to delay".format(model_ols.params['jobs_running']))
print("- Each additional core adds ~{:.2f} seconds to delay".format(model_ols.params['num_cores']))

# Calculate standardized coefficients to compare relative importance
X_standardized = (X - X.mean()) / X.std()
X_std_const = sm.add_constant(X_standardized)
model_std = sm.OLS(y, X_std_const).fit()

print("\n" + "-" * 80)
print("RELATIVE IMPORTANCE (Standardized Coefficients):")
print("-" * 80)
for var, coef in zip(['jobs_ahead', 'jobs_running', 'num_cores'], model_std.params[1:]):
    print(f"{var:15s}: {coef:8.2f} (magnitude indicates relative importance)")

most_important = X.columns[np.argmax(np.abs(model_std.params[1:]))]
print(f"\nMost predictive factor: {most_important}")

# Correlation analysis
print("\n" + "=" * 80)
print("CORRELATION MATRIX")
print("=" * 80)
corr_df = df[['queuing_delay', 'jobs_ahead', 'jobs_running', 'num_cores']].corr()
print(corr_df)

# Visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Scatter plots with regression lines
for idx, (var, ax) in enumerate(zip(['jobs_ahead', 'jobs_running', 'num_cores'], axes[0])):
    ax.scatter(df[var], df['queuing_delay'], alpha=0.5, s=30)
    
    # Add regression line
    z = np.polyfit(df[var], df['queuing_delay'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[var].min(), df[var].max(), 100)
    ax.plot(x_line, p(x_line), "r--", linewidth=2, label=f'y={z[0]:.2f}x+{z[1]:.2f}')
    
    ax.set_xlabel(var.replace('_', ' ').title(), fontsize=12)
    ax.set_ylabel('Queuing Delay (seconds)', fontsize=12)
    ax.set_title(f'Delay vs {var.replace("_", " ").title()}', fontsize=13)
    ax.legend()
    ax.grid(True, alpha=0.3)

# 4. Actual vs Predicted
axes[1, 0].scatter(y, model_ols.predict(X_with_const), alpha=0.5)
axes[1, 0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
axes[1, 0].set_xlabel('Actual Delay (seconds)', fontsize=12)
axes[1, 0].set_ylabel('Predicted Delay (seconds)', fontsize=12)
axes[1, 0].set_title('Actual vs Predicted Queuing Delay', fontsize=13)
axes[1, 0].grid(True, alpha=0.3)

# 5. Residuals plot
residuals = y - model_ols.predict(X_with_const)
axes[1, 1].scatter(model_ols.predict(X_with_const), residuals, alpha=0.5)
axes[1, 1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1, 1].set_xlabel('Predicted Delay (seconds)', fontsize=12)
axes[1, 1].set_ylabel('Residuals (seconds)', fontsize=12)
axes[1, 1].set_title('Residual Plot', fontsize=13)
axes[1, 1].grid(True, alpha=0.3)

# 6. Feature importance bar chart
importance_data = pd.DataFrame({
    'Feature': ['jobs_ahead', 'jobs_running', 'num_cores'],
    'Standardized Coefficient': np.abs(model_std.params[1:])
}).sort_values('Standardized Coefficient', ascending=True)

axes[1, 2].barh(importance_data['Feature'], importance_data['Standardized Coefficient'], color='steelblue')
axes[1, 2].set_xlabel('Absolute Standardized Coefficient', fontsize=12)
axes[1, 2].set_title('Relative Feature Importance', fontsize=13)
axes[1, 2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('figs/queue_state_impact_analysis.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Additional analysis: By core type
print("\n" + "=" * 80)
print("ANALYSIS BY CORE CONFIGURATION")
print("=" * 80)

for cores in [4, 16, 64]:
    df_cores = df[df['num_cores'] == cores]
    if len(df_cores) > 0:
        X_cores = df_cores[['jobs_ahead', 'jobs_running']]
        y_cores = df_cores['queuing_delay']
        X_cores_const = sm.add_constant(X_cores)
        model_cores = sm.OLS(y_cores, X_cores_const).fit()
        
        print(f"\n{cores}-Core Jobs:")
        print(f"  R-squared: {model_cores.rsquared:.4f}")
        print(f"  jobs_ahead coefficient: {model_cores.params['jobs_ahead']:.2f} seconds")
        print(f"  jobs_running coefficient: {model_cores.params['jobs_running']:.2f} seconds")

print("\n" + "=" * 80)

In [ ]:
# Sort by submission time
df = df.sort_values('submitted_dt').reset_index(drop=True)

# Define colors for different core counts
core_colors = {4: '#3498db', 16: '#e74c3c', 64: '#2ecc71'}
core_labels = {4: '4 cores', 16: '16 cores', 64: '64 cores'}

# Create figure with multiple subplots
fig = plt.figure(figsize=(20, 12))

# Queue state over time
ax1 = plt.subplot(2, 1, 1)

# Plot jobs_ahead and jobs_running over time
ax1.scatter(df['submitted_dt'], df['jobs_ahead'], label='Jobs Ahead', alpha=0.6, s=40, color='#e74c3c')
ax1.scatter(df['submitted_dt'], df['jobs_running'], label='Jobs Running', alpha=0.6, s=40, color='#3498db')
ax1.scatter(df['submitted_dt'], df['jobs_in_my_queue'], label='Jobs in My Queue', alpha=0.6, s=40, color='#9b59b6')

ax1.set_xlabel('Time', fontsize=12)
ax1.set_ylabel('Number of Jobs', fontsize=12)
ax1.set_title('Queue State at Job Submission Time', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Queuing delay over time
ax2 = plt.subplot(2, 1, 2)

for cores in sorted(core_colors.keys()):
    df_cores = df[df['num_cores'] == cores]
    ax2.scatter(df_cores['submitted_dt'], df_cores['queuing_delay'], 
                label=core_labels[cores], alpha=0.6, s=60, color=core_colors[cores],
                edgecolors='black', linewidth=0.5)
    
    # Add trend line
    if len(df_cores) > 1:
        x_numeric = (df_cores['submitted_dt'] - df_cores['submitted_dt'].min()).dt.total_seconds()
        z = np.polyfit(x_numeric, df_cores['queuing_delay'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(x_numeric.min(), x_numeric.max(), 100)
        time_line = df_cores['submitted_dt'].min() + pd.to_timedelta(x_line, unit='s')
        ax2.plot(time_line, p(x_line), '--', color=core_colors[cores], linewidth=2, alpha=0.8)

ax2.set_xlabel('Time', fontsize=12)
ax2.set_ylabel('Queuing Delay (seconds)', fontsize=12)
ax2.set_title('Queuing Delay Over Time by Core Count', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('figs/queue_timeline_plot.pdf', dpi=300, bbox_inches='tight')
plt.show()


# Print summary statistics
# print("=" * 80)
# print("TIMELINE SUMMARY STATISTICS")
# print("=" * 80)
# print(f"\nTotal jobs analyzed: {len(df)}")
# print(f"Time range: {df['submitted_dt'].min()} to {df['started_dt'].max()}")
# print(f"Total duration: {(df['started_dt'].max() - df['submitted_dt'].min()).total_seconds() / 3600:.2f} hours")

# print("\nQueuing Delay Statistics by Core Count:")
# print("-" * 80)
# for cores in sorted(core_colors.keys()):
#     df_cores = df[df['num_cores'] == cores]
#     print(f"\n{cores} cores ({len(df_cores)} jobs):")
#     print(f"  Mean delay: {df_cores['queuing_delay'].mean():.2f} seconds")
#     print(f"  Median delay: {df_cores['queuing_delay'].median():.2f} seconds")
#     print(f"  Min delay: {df_cores['queuing_delay'].min():.2f} seconds")
#     print(f"  Max delay: {df_cores['queuing_delay'].max():.2f} seconds")

# print("\n" + "=" * 80)

In [ ]:
# Create a detailed timeline for a subset of jobs (first 50 jobs for clarity)
fig2, ax = plt.subplots(figsize=(20, 10))

n_jobs = min(50, len(df))
df_subset = df.head(n_jobs)

y_positions = range(n_jobs)

for idx, (i, row) in enumerate(df_subset.iterrows()):
    color = core_colors[row['num_cores']]
    
    # Draw waiting bar
    start_x = row['submitted_dt']
    end_x = row['started_dt']
    
    ax.plot([start_x, end_x], [idx, idx], color=color, linewidth=8, alpha=0.7, solid_capstyle='butt')
    
    # Submission marker
    ax.scatter(start_x, idx, color=color, s=100, zorder=5, marker='o', edgecolors='black', linewidth=2)
    
    # Start marker
    ax.scatter(end_x, idx, color=color, s=100, zorder=5, marker='s', edgecolors='black', linewidth=2)
    
    # Add job label
    ax.text(start_x, idx, f"  {row['file'][:15]}...", va='center', fontsize=8, alpha=0.7)

ax.set_xlabel('Time', fontsize=14, fontweight='bold')
ax.set_ylabel('Job Index', fontsize=14, fontweight='bold')
ax.set_title(f'Detailed Timeline: First {n_jobs} Jobs (○=submitted, □=started)', fontsize=16, fontweight='bold')
ax.set_ylim(-1, n_jobs)
ax.grid(True, alpha=0.3, axis='x')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Legend
legend_elements = [
    mpatches.Patch(facecolor=core_colors[cores], edgecolor='black', label=core_labels[cores]) 
    for cores in sorted(core_colors.keys())
]
legend_elements.extend([
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=10, 
               markeredgecolor='black', markeredgewidth=2, label='Submitted'),
    plt.Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', markersize=10, 
               markeredgecolor='black', markeredgewidth=2, label='Started')
])
ax.legend(handles=legend_elements, loc='upper right', fontsize=11, framealpha=0.9)

plt.tight_layout()
plt.savefig('figs/queue_timeline_combined_detailed.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df = df.sort_values('submitted_dt').reset_index(drop=True)

# Define colors for different core counts
core_colors = {4: '#3498db', 16: '#e74c3c', 64: '#2ecc71'}
core_labels = {4: '4 cores', 16: '16 cores', 64: '64 cores'}

def plot_timeline_by_cores(df, num_cores, max_jobs=50):
    """
    Create a detailed timeline plot for jobs with a specific core count.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The full dataframe with job data
    num_cores : int
        Number of cores to filter by (4, 16, or 64)
    max_jobs : int
        Maximum number of jobs to display (default: 50)
    """
    # Filter by core count
    df_filtered = df[df['num_cores'] == num_cores].copy()
    
    if len(df_filtered) == 0:
        print(f"No jobs found with {num_cores} cores!")
        return
    
    # Take first max_jobs
    n_jobs = min(max_jobs, len(df_filtered))

    df_subset = df_filtered[200:300]
    # df_subset = df_filtered.head(n_jobs)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(20, max(10, n_jobs * 0.2)))
    
    color = core_colors[num_cores]
    
    for idx, (i, row) in enumerate(df_subset.iterrows()):
        # Draw waiting bar
        start_x = row['submitted_dt']
        end_x = row['started_dt']
        
        ax.plot([start_x, end_x], [idx, idx], color=color, linewidth=8, alpha=0.7, solid_capstyle='butt')
        
        # Submission marker
        ax.scatter(start_x, idx, color=color, s=120, zorder=5, marker='o', edgecolors='black', linewidth=2)
        
        # Start marker
        ax.scatter(end_x, idx, color=color, s=120, zorder=5, marker='s', edgecolors='black', linewidth=2)
        
        # Add job label and delay info
        delay_text = f"{row['queuing_delay']:.1f}s"
        ax.text(start_x, idx, f"  {row['file'][:20]}... ({delay_text})", 
                va='center', fontsize=9, alpha=0.8)
    
    ax.set_xlabel('Time', fontsize=14, fontweight='bold')
    ax.set_ylabel('Job Index', fontsize=14, fontweight='bold')
    ax.set_title(f'Detailed Timeline: {num_cores}-Core Jobs (First {n_jobs} jobs) - ○=submitted, □=started', 
                 fontsize=16, fontweight='bold')
    ax.set_ylim(-1, n_jobs)
    ax.grid(True, alpha=0.3, axis='x')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Legend
    legend_elements = [
        mpatches.Patch(facecolor=color, edgecolor='black', label=core_labels[num_cores]),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=10, 
                   markeredgecolor='black', markeredgewidth=2, label='Submitted'),
        plt.Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', markersize=10, 
                   markeredgecolor='black', markeredgewidth=2, label='Started'),
        plt.Line2D([0], [0], color=color, linewidth=8, alpha=0.7, label='Waiting Period')
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=11, framealpha=0.9)
    
    plt.tight_layout()
    plt.savefig(f'figs/timeline_{num_cores}_cores.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    
# ============================================================================
# USAGE: Choose which core count to display
# ============================================================================

# Option 1: Display 4-core jobs
# plot_timeline_by_cores(df, num_cores=4, max_jobs=50)

# Option 2: Display 16-core jobs
# plot_timeline_by_cores(df, num_cores=16, max_jobs=50)

# Option 3: Display 64-core jobs
# plot_timeline_by_cores(df, num_cores=64, max_jobs=50)

# You can also adjust the max_jobs parameter:
# plot_timeline_by_cores(df, num_cores=16, max_jobs=30)  # Show only 30 jobs

# ============================================================================
# Or create all three plots at once (comment out above and uncomment below)
# ============================================================================

# for cores in [4, 16, 64]:
#     plot_timeline_by_cores(df, num_cores=cores, max_jobs=20)

In [ ]:
for num_cores in [4, 16, 64]:
    df_filtered = df[df['num_cores'] == num_cores].copy()

    # Extract day of week (0=Monday, 6=Sunday) and time of day
    df_filtered['day_of_week'] = df_filtered['submitted_dt'].dt.dayofweek
    df_filtered['time_of_day'] = df_filtered['submitted_dt'].dt.hour + df_filtered['submitted_dt'].dt.minute / 60

    # Remove outliers using IQR method
    Q1 = df_filtered['queuing_delay'].quantile(0.25)
    Q3 = df_filtered['queuing_delay'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Filter out outliers
    df_filtered = df_filtered[(df_filtered['queuing_delay'] >= lower_bound) & (df_filtered['queuing_delay'] <= upper_bound)].copy()
    outliers_removed = len(df_filtered) - len(df_filtered)

    # Create bins for day of week and time intervals
    filter_time_in_minutes = 60

    df_filtered['time_in_minutes'] = df_filtered['time_of_day'] * 60
    df_filtered['time_bin'] = (df_filtered['time_in_minutes'] // filter_time_in_minutes) * filter_time_in_minutes  # Round down to nearest 5 minutes
    df_filtered['day_time_bin'] = df_filtered['day_of_week'].astype(str) + '_' + df_filtered['time_bin'].astype(str)

    # Group by day and time bin, then average the queuing delay
    df_averaged = df_filtered.groupby(['day_of_week', 'time_bin']).agg({
        'queuing_delay': 'mean'
    }).reset_index()

    # Calculate x_position: day + time as fraction of day
    df_averaged['time_of_day'] = df_averaged['time_bin'] / 60  # Convert minutes back to hours
    df_averaged['x_position'] = df_averaged['day_of_week'] + df_averaged['time_of_day'] / 24

    # Create the scatter plot
    plt.figure(figsize=(14, 7))
    plt.scatter(df_averaged['x_position'], df_averaged['queuing_delay'], 
            alpha=0.6, s=50, c='steelblue', edgecolors='black', linewidth=0.5,
            label=f'Avg Queuing Delay ({filter_time_in_minutes}-min bins)')

    plt.plot(df_averaged['x_position'], df_averaged['queuing_delay'], 
            alpha=0.6, c='red', linewidth=1, label='Avg Queuing Delay')

    plt.xlabel('Day of Week', fontsize=12)
    plt.ylabel('Queuing Delay (seconds)', fontsize=12)
    plt.title('Queuing Delay by Day of Week', fontsize=14, fontweight='bold')

    plt.legend(loc='best', fontsize=10)

    # Set x-axis to show days of the week
    day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    plt.xlim(-0.5, 6.5)
    plt.xticks(range(7), day_names, rotation=45, ha='right')

    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

In [ ]:
for num_cores in [4, 16, 64]:
    df_filtered = df[df['num_cores'] == num_cores].copy()

    # Extract day of week (0=Monday, 6=Sunday) and time of day
    df_filtered['time_of_day'] = df_filtered['submitted_dt'].dt.hour + df_filtered['submitted_dt'].dt.minute / 60

    # Remove outliers using IQR method
    Q1 = df_filtered['queuing_delay'].quantile(0.25)
    Q3 = df_filtered['queuing_delay'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Filter out outliers
    df_filtered = df_filtered[(df_filtered['queuing_delay'] >= lower_bound) & (df_filtered['queuing_delay'] <= upper_bound)].copy()
    outliers_removed = len(df_filtered) - len(df_filtered)

    # Create bins for day of week and time intervals
    filter_time_in_minutes = 10

    df_filtered['time_in_minutes'] = df_filtered['time_of_day'] * 60
    df_filtered['time_bin'] = (df_filtered['time_in_minutes'] // filter_time_in_minutes) * filter_time_in_minutes  # Round down to nearest 5 minutes

    # Group by day and time bin, then average the queuing delay
    df_averaged = df_filtered.groupby(['time_bin']).agg({
        'queuing_delay': 'mean'
    }).reset_index()

    # Calculate x_position: day + time as fraction of day
    df_averaged['time_of_day'] = df_averaged['time_bin'] / 60  # Convert minutes back to hours

    # Create the scatter plot
    plt.figure(figsize=(14, 7))
    plt.scatter(df_averaged['time_of_day'], df_averaged['queuing_delay'], 
            alpha=0.6, s=50, c='steelblue', edgecolors='black', linewidth=0.5,
            label=f'Avg Queuing Delay ({filter_time_in_minutes}-min bins)')

    plt.plot(df_averaged['time_of_day'], df_averaged['queuing_delay'], 
            alpha=0.6, c='red', linewidth=1, label='Avg Queuing Delay')

    plt.xlabel('Time of Day (24-hour)', fontsize=12)
    plt.ylabel('Queuing Delay (seconds)', fontsize=12)
    plt.title('Queuing Delay by Time of Day', fontsize=14, fontweight='bold')

    plt.legend(loc='best', fontsize=10)

    # Set x-axis to show hours from 0 to 24
    plt.xlim(0, 24)
    plt.xticks(ticks = np.arange(0,24), labels = [am_pm_label(i) for i in range(24)], rotation=45)

    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()